# Exploration of COLET dataset

The original dataset from this [paper](https://doi.org/10.1016/j.cmpb.2022.106989) has a single *.mat* file containing tabular data from the experiments.

Since this format is not easy to use with *Python*, a MATLAB script was used to extract the relevant data as *.csv* files, which can be easily loaded with *Pandas* library.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

COLET_DATASET_DIR = Path("../datasets/COLET_CSV")
EXP_SELECT = "participant_01/Task_1"

In [42]:
# Load the different daat from eye tracker
gaze_df = pd.read_csv(COLET_DATASET_DIR / EXP_SELECT / "gaze.csv")
pupil_df = pd.read_csv(COLET_DATASET_DIR / EXP_SELECT / "pupil.csv")
blink_df = pd.read_csv(COLET_DATASET_DIR / EXP_SELECT / "blinks.csv")

print(f"Loaded {gaze_df.shape[0]} gaze records:")
display(gaze_df.head(1))
print(f"Loaded {pupil_df.shape[0]} pupil records:")
display(pupil_df.head(5))
print(f"Loaded {blink_df.shape[0]} blink records:")
display(blink_df.head())

Loaded 8205 gaze records:


,gaze_timestamp,confidence,norm_pos_x,norm_pos_y,gaze_point_3d_x,gaze_point_3d_y,gaze_point_3d_z
0,5410.551715,0.999499,0.446264,0.846886,-6.72063,-31.252881,80.700682


Loaded 16406 pupil records:


,pupil_timestamp,eye_id,confidence,norm_pos_x,norm_pos_y,diameter,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b
0,5410.553797,1,1.000000,0.406205,0.708513,40.757561,77.991356,55.965595,25.383852,40.757561
1,5410.553797,1,0.999646,0.406235,0.708578,40.764095,77.997212,55.953014,26.110611,40.764095
2,5410.557872,0,0.999661,0.326001,0.395972,38.729047,62.592283,115.973389,23.373748,38.729047
3,5410.557872,0,1.000000,0.325980,0.396084,38.723305,62.588215,115.951805,23.741541,38.723305
4,5410.561675,1,1.000000,0.406046,0.707954,40.842903,77.960808,56.072765,25.485304,40.842903


Loaded 2 blink records:


,id,start_timestamp,duration,end_timestamp,start_frame_index,index,end_frame_index,confidence,filter_response,base_data
0,1,5437.625617,0.236131,5437.861748,37,40,44,0.703872,0.5068225043614704 0.5512669488059149 0.595711...,5437.625617 5437.629564 5437.633621 5437.63786...
1,2,5444.161561,0.180073,5444.341634,231,233,236,0.553669,0.5048782729116744 0.5493227173561188 0.592878...,5444.161561 5444.165642 5444.169793 5444.17554...


In [12]:
# Some insights about the duration of the gaze and pupil data
gaze_start_time = gaze_df['gaze_timestamp'].min()
gaze_end_time = gaze_df['gaze_timestamp'].max()
pupil_start_time = pupil_df['pupil_timestamp'].min()
pupil_end_time = pupil_df['pupil_timestamp'].max()
print(f"Gaze data duration: {gaze_end_time - gaze_start_time} s")
print(f"Pupil data duration: {pupil_end_time - pupil_start_time} s")

print(f"Estimated gaze frequency: {gaze_df.shape[0] / (gaze_end_time - gaze_start_time):.2f} Hz")
print(f"Estimated pupil (right) frequency: {pupil_df[pupil_df.eye_id == 1].shape[0] / (pupil_end_time - pupil_start_time):.2f} Hz")
print(f"Estimated pupil (left) frequency: {pupil_df[pupil_df.eye_id == 0].shape[0] / (pupil_end_time - pupil_start_time):.2f} Hz")


Gaze data duration: 33.80600950000007 s
Pupil data duration: 33.80392700000084 s
Estimated gaze frequency: 242.71 Hz
Estimated pupil (right) frequency: 242.69 Hz
Estimated pupil (left) frequency: 242.63 Hz


It looks like they are using two methods to calculate the pupil related metrics ('2D c++' / '3D c++'). It also seems that there is always one that has a higher confidence, so the diameter will be chosen from the highest confidence value.

Also, it seems like it is always switching between left and right eye for a **combined** frequency of 240Hz

In [30]:
# Sort pupil data by choosing best confidence for each timestamp for each eye
pupil_df_best = pupil_df.groupby(['pupil_timestamp', 'eye_id'], as_index=False).apply(lambda x: x.loc[x['confidence'].idxmax()])
pupil_df_best.reset_index(drop=True, inplace=True)
pupil_df_best.drop(columns=['norm_pos_x', 'norm_pos_y'], inplace=True)
pupil_df_best.rename(columns={'diameter': 'pupil_diameter', 'confidence': 'pupil_confidence'}, inplace=True)
pupil_df_best

,pupil_timestamp,eye_id,pupil_confidence,pupil_diameter,ellipse_center_x,ellipse_center_y,ellipse_axis_a,ellipse_axis_b
0,5410.553797,1,1.000000,40.757561,77.991356,55.965595,25.383852,40.757561
1,5410.557872,0,1.000000,38.723305,62.588215,115.951805,23.741541,38.723305
2,5410.561675,1,1.000000,40.842903,77.960808,56.072765,25.485304,40.842903
3,5410.565491,0,1.000000,39.237560,62.568382,115.802177,23.588182,39.237560
4,5410.571092,1,1.000000,40.950336,77.953743,56.054668,25.205196,40.950336
...,...,...,...,...,...,...,...,...
8198,5444.341634,0,0.735939,30.008591,71.342831,105.565708,22.186894,30.008591
8199,5444.345790,1,0.045619,40.353993,58.252892,55.428499,11.294928,40.353993
8200,5444.350231,0,0.390900,41.681687,66.234522,113.613751,27.634551,41.681687
8201,5444.353666,1,0.657261,42.941853,79.710823,60.829433,27.937931,42.941853


In [57]:
# Merge gaze and pupil data on closest timestamp match
merged_df = pd.merge_asof(gaze_df.sort_values('gaze_timestamp'),
                          pupil_df_best.sort_values('pupil_timestamp'),
                          left_on='gaze_timestamp',
                          right_on='pupil_timestamp',
                          direction='nearest',
                          tolerance=1.0/240)  # tolerance in ms
merged_df[merged_df.pupil_confidence.isna()].count()

gaze_timestamp      129
confidence          129
norm_pos_x          129
norm_pos_y          129
gaze_point_3d_x     129
gaze_point_3d_y     129
gaze_point_3d_z     129
pupil_timestamp       0
eye_id                0
pupil_confidence      0
pupil_diameter        0
ellipse_center_x      0
ellipse_center_y      0
ellipse_axis_a        0
ellipse_axis_b        0
dtype: int64